# Strategy - Mean Reversion

In [ ]:
# ============================================================
# S03 - Mean Reversion
# ============================================================

import xarray as xr
import numpy as np

import qnt.data as qndata
import qnt.ta as qnta
import qnt.output as qnout

# ============================================================
# Load Data
# ============================================================

data = qndata.cryptodaily_load_data(
    min_date="2015-01-01"
)

# ============================================================
# Strategy
# ============================================================

def strategy(data):

    close = data.sel(field="close")
    is_liquid = data.sel(field="is_liquid")

    # --------------------------------------------------------
    # Daily Returns
    # --------------------------------------------------------

    ret = close / close.shift(time=1) - 1

    vol20 = qnta.std(ret, 20)

    # --------------------------------------------------------
    # Mean Reversion Signal
    # --------------------------------------------------------

    ret5 = close / close.shift(time=5) - 1

    # Recent losers preferred
    score = (-ret5).rank("asset")

    score = score * is_liquid

    # --------------------------------------------------------
    # Top Assets
    # --------------------------------------------------------

    ranks = score.rank("asset")

    top_assets = xr.where(
        ranks >= 6,
        1,
        0
    )

    weights = score * top_assets

    # --------------------------------------------------------
    # Inverse Vol Weighting
    # --------------------------------------------------------

    inv_vol = 1 / (vol20 + 1e-6)

    weights = weights * inv_vol

    # --------------------------------------------------------
    # Normalize
    # --------------------------------------------------------

    denom = weights.sum("asset")

    weights = xr.where(
        denom > 0,
        weights / denom,
        0
    )

    return weights.fillna(0)

# ============================================================
# Generate Weights
# ============================================================

weights = strategy(data)

# ============================================================
# Clean
# ============================================================

weights = qnout.clean(
    weights,
    data,
    "crypto_daily_long"
)

# ============================================================
# Output
# ============================================================

qnout.write(weights)

100% (15259192 of 15259192) |############| Elapsed Time: 0:00:00 Time:  0:00:000:00
Output cleaning...
Fix unique timestamps
Forward filling missing prices...
Check liquidity...
Ok.
Check for missed dates...
Ok.
Check positive positions...
Ok.
Final normalization...
Output cleaning complete.
Write output: /root/fractions.nc.gz

In [ ]:
# ============================================================
# Strategy Statistics
# ============================================================

import qnt.stats as qnstats

stats = qnstats.calc_stat(
    data,
    weights.sel(time=slice("2016-01-01", None))
)

stats_pd = stats.to_pandas()

print(stats_pd.tail())

print("\nFinal Metrics:")

print(
    stats_pd.iloc[-1][[
        "equity",
        "sharpe_ratio",
        "max_drawdown",
        "avg_turnover",
        "avg_holding_time"
    ]]
)

field         equity  relative_return  volatility  underwater  max_drawdown  \
time                                                                          
2026-06-05  0.735644        -0.067771    0.785655   -0.979277     -0.979277   
2026-06-06  0.725840        -0.013328    0.785565   -0.979554     -0.979554   
2026-06-07  0.775667         0.068648    0.785742   -0.978150     -0.979554   
2026-06-08  0.784025         0.010776    0.785645   -0.977915     -0.979554   
2026-06-09  0.758754        -0.032233    0.785609   -0.978626     -0.979554   

field       sharpe_ratio  mean_return  bias  instruments  avg_turnover  \
time                                                                     
2026-06-05      0.363292     0.285422   1.0         37.0      0.303344   
2026-06-06      0.361613     0.284071   1.0         37.0      0.303340   
2026-06-07      0.369804     0.290571   1.0         37.0      0.303325   
2026-06-08      0.371066     0.291526   1.0         37.0      0.303330   
2026-06-09      0.367059     0.288364   1.0         37.0      0.303350   

field       avg_holding_time  
time                          
2026-06-05          6.945545  
2026-06-06          6.946485  
2026-06-07          6.947062  
2026-06-08          6.948210  
2026-06-09          6.950480  

Final Metrics:
field
equity              0.758754
sharpe_ratio        0.367059
max_drawdown       -0.979554
avg_turnover        0.303350
avg_holding_time    6.950480
Name: 2026-06-09 00:00:00, dtype: float64